# Silver Layer Transformation 

In the Silver layer, the Bronze datasets are cleaned and standardized to ensure data consistency and usability for downstream analytics. String fields are trimmed, special characters like | are removed, and placeholder values such as N/A, NA, or empty strings are replaced with NULL. Country names are normalized; for example, UK, U.K., and misspelled variants like United Kindom are standardized to United Kingdom. Monetary fields like unit_price, cost, and sales are converted from strings with currency symbols into numeric values, while a separate currency column stores the currency type. Duplicate records are eliminated using the appropriate primary keys, and an ingestion_timestamp column is added to record when the data was processed in the Silver layer.

# Silver Table (Region)

In [0]:
%sql
SELECT * FROM project.bronze.region;

In [0]:
# Load bronze.region table
df_region = spark.table("project.bronze.region")
display(df_region)
df_region.printSchema()

In [0]:
%sql
-- Check that 'sales_territory_key' is unique
--Output should be empty
SELECT 
    sales_territory_key, 
    COUNT(*) AS cnt
FROM project.bronze.region
GROUP BY sales_territory_key
HAVING cnt > 1;

In [0]:
from pyspark.sql import functions as F

# Trim string columns
string_cols = [c for c, t in df_region.dtypes if t == "string"]
for c in string_cols:
    df_region.withColumn(c, 
                            F.when(F.upper(c).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
                            .otherwise(F.col(c)))  \
             .withColumn(c, F.trim(F.col(c)))

# Replace 'USA' with 'United States'
df_region = df_region.withColumn(
    "country",
    F.when(F.upper(F.col("country")) == F.lit("USA"), "United States")
     .otherwise(F.col("country"))
)

# Drop ingestion timestamp column
df_region = df_region.drop("source_file", "ingestion_timestamp")

# Sanity check
display(df_region)

In [0]:
# Write silver region table
(df_region.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.region")
)

# Sanity check
display(spark.table("project.silver.region"))

# Silver Table (Salesperson_region)

In [0]:
%sql 
SELECT * FROM project.bronze.salesperson_region;

In [0]:
df_salesperson_region = spark.table("project.bronze.salesperson_region")
display(df_salesperson_region)
df_salesperson_region.printSchema()

In [0]:
%sql
-- Check if there are any null values in the keys
SELECT *
FROM project.bronze.salesperson_region
WHERE employee_key IS NULL 
   OR sales_territory_key IS NULL;

In [0]:
%sql
-- Check if 'employee_key' is unique
-- This counts how many territories are assigned to each employee
SELECT 
    employee_key, 
    COUNT(sales_territory_key) AS territory_cnt
FROM project.bronze.salesperson_region
GROUP BY employee_key;

In [0]:
%sql
-- Check if {employee_key, sales_territory_key} is unique
-- If no output -> {employee_key, sales_territory_key} = Primary Key
SELECT 
    employee_key, 
    sales_territory_key, 
    COUNT(*) AS duplicate_cnt
FROM project.bronze.salesperson_region
GROUP BY employee_key, sales_territory_key
HAVING COUNT(*) > 1;

In [0]:
# Drop unwanted columns
df_salesperson_region = df_salesperson_region.drop("source_file", "ingestion_timestamp")

# Write silver salesperson_region table
(df_salesperson_region.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.salesperson_region")
)

# Sanity check
display(spark.table("project.silver.salesperson_region"))

# Silver Table (Salesperson)

In [0]:
%sql
SELECT * FROM project.bronze.salesperson;

In [0]:
df_salesperson = spark.table("project.bronze.salesperson")
display(df_salesperson)
df_salesperson.printSchema()

In [0]:
# Drop unwanted columns
df_salesperson = df_salesperson.drop("source_file", "ingestion_timestamp")

# Strip and capitalize string columns
string_cols = [c for c, t in df_salesperson.dtypes if t == "string"]
for c in string_cols:
    df_salesperson = df_salesperson.withColumn(c, 
                                         F.when(F.upper(c).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
                                            .otherwise(F.col(c)))  \
                                   .withColumn(c, F.trim(F.col(c))) \
                                   .withColumn(c, F.initcap(F.col(c)))

In [0]:
%sql
-- Check total count and compare to distinct 'employee_key' and distinct 'employee_id' counts
SELECT
    COUNT(*) AS total_cnt,
    COUNT(DISTINCT employee_key) AS distinct_empl_key_cnt,
    COUNT(DISTINCT employee_id) AS distinct_empl_id_cnt
FROM project.bronze.salesperson;

In [0]:
%sql
--Check that 'employee_key' and 'employee_id' are not null
--Count should be zero
SELECT
    COUNT(*) AS key_id_null_cnt
FROM project.bronze.salesperson
WHERE employee_key IS NULL 
   OR employee_id IS NULL;

In [0]:
# Write silver salesperson table
(df_salesperson.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.salesperson")
)

# Sanity check
display(spark.table("project.silver.salesperson"))

# Silver Table (Reseller)

In [0]:
%sql
SELECT * FROM project.bronze.reseller;

In [0]:
df_reseller = spark.table("project.bronze.reseller")
display(df_reseller)
df_reseller.printSchema()

In [0]:
# Drop unwanted columns
df_reseller = df_reseller.drop("source_file", "ingestion_timestamp")

# Strip and capitalize string columns
string_cols = [c for c, t in df_reseller.dtypes if t == "string"]
for c in string_cols:
    df_reseller = df_reseller.withColumn(c, 
                                         F.when(F.upper(c).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
                                            .otherwise(F.col(c))) \
                             .withColumn(c, F.trim(F.col(c))) \
                             .withColumn(c, F.initcap(F.col(c)))

In [0]:
%sql
-- Check distinct state provinces
SELECT DISTINCT state_province 
FROM project.bronze.reseller;

In [0]:
%sql
-- Check distinct country regions
SELECT DISTINCT country_region 
FROM project.bronze.reseller;

In [0]:
# Replace country abbreviations with country name
# We have already capitalize 'country_region'
df_reseller = df_reseller.replace(
    {"Usa": "United States",
     "Us": "United States",
     "U.s.a.": "United States",
     "U.k.": "United Kingdom",
     "Uk": "United Kingdom"},
    subset=["country_region"]
)

# Sanity check
display(df_reseller.select("country_region").distinct())

In [0]:
%sql
-- Check if 'reseller_key' is null. 
--Count should be zero.
SELECT COUNT(*) AS null_reseller_key_cnt
FROM project.bronze.reseller
WHERE reseller_key IS NULL;



In [0]:
%sql
-- Check if 'reseller_key' is unique. Result should be empty.
SELECT
    reseller_key,
    COUNT(*) AS cnt
FROM project.bronze.reseller
GROUP BY reseller_key
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Check that 'sales_territory_key' is unique
-- If the result is empty, the key is a valid Primary Key
SELECT 
    COUNT(*) AS total_cnt,
    COUNT(DISTINCT reseller_key) AS distinct_reseller_key_cnt
FROM project.bronze.reseller;

In [0]:
# Write silver reseller table
(df_reseller.write
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .format("delta")
            .saveAsTable("project.silver.reseller")
)

# Sanity check
display(spark.table("project.silver.reseller"))

# Silver Table (Targets)

In [0]:
%sql
SELECT * FROM project.bronze.targets;

In [0]:
df_targets = spark.table("project.bronze.targets")
display(df_targets)
df_targets.printSchema()

In [0]:
from pyspark.sql import functions as F

# Drop unwanted columns
df_targets = df_targets.drop("source_file", "ingestion_timestamp")

# Trim string columns
df_targets = (df_targets.withColumn("target", F.trim("target"))
                        .withColumn("target_month", F.trim("target_month"))
                        )


In [0]:
%sql
-- Check currency codes available in 'target' column
-- Check both start and end of string
SELECT 
    DISTINCT LEFT(target, 1) AS symbol_start,
    RIGHT(target, 1) AS symbol_end
FROM project.bronze.targets;

In [0]:
%sql
-- Scan for NULL values in the 'target' column
-- If results are found, these records will need imputation or filtering in Silver
SELECT *
FROM project.bronze.targets
WHERE target IS NULL;

In [0]:
# Map currency sign to currency code
df_targets = df_targets.withColumn("currency",
                                F.when(F.col("target").rlike("\\$"), F.lit("USD"))
                                .when(F.col("target").rlike("€"), F.lit("EUR"))
                                .when(F.col("target").rlike("£"), F.lit("GBP"))
                                .otherwise(F.lit("USD"))
                            )

# Clean 'target' column and cast to decimal
df_targets = df_targets.withColumn("target",
    F.regexp_replace(F.col("target"), r"[$,€£\s]", "").cast("decimal(18,2)"))

In [0]:
%sql
-- Check for NULL values in the 'target_month' column
-- Result should be zero to ensure time-series integrity
SELECT COUNT(*) AS null_target_month_cnt
FROM project.bronze.targets
WHERE target_month IS NULL;


In [0]:
%sql
-- Inspect unique 'target_month' values to identify formatting patterns
-- This helps in designing the TO_DATE parsing logic in the Silver layer
SELECT DISTINCT target_month
FROM project.bronze.targets;

In [0]:
# Convert string month to date dtype month
df_targets = df_targets.withColumn("targetmonth",
                            F.regexp_replace("target_month", "^[A-Za-z]+,\\s*", "")) \
                        .withColumn("target_month_clean",
                            F.coalesce(
                                F.to_date("targetmonth", "MMMM d, yyyy"),
                                F.to_date("targetmonth", "MMM d, yyyy")
                            ))

# Sanity check
display(df_targets)

In [0]:
%sql
-- Check if employee ID is null
-- Any count > 0 indicates records that cannot be linked to the Salesperson dimension
SELECT COUNT(*) AS null_employee_id_cnt 
FROM project.bronze.targets
WHERE employee_id IS NULL;

In [0]:
# Count of targets per 'employee_id'
display(spark.sql("""
                  SELECT employee_id, COUNT(target) AS target_cnt
                  FROM project.bronze.targets
                  GROUP BY employee_id
                  ORDER BY target_cnt DESC
                  """))

In [0]:
# Check for duplicate records
df_targets.groupBy(df_targets.columns).count().filter(F.col("count") > 1).show()

In [0]:
# Keep columns to save
df_targets = df_targets.drop("target_month", "targetmonth") \
                       .withColumnRenamed("target_month_clean", "target_month") \
                       .select("employee_id", "target", "currency", "target_month")

# Write silver table
(
    df_targets.write
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .format("delta")
          .saveAsTable("project.silver.targets")
)

# Sanity check
display(spark.table("project.silver.targets"))

# ↓ **INSERT HERE** ↓

In [0]:
%sql
SELECT * FROM project.bronze.product LIMIT 10;

In [0]:
%sql
-- Check for duplicate product keys in the Bronze layer 
-- (Identifies Primary Key violations before processing)
SELECT 
    product_key, 
    COUNT(*) AS cnt
FROM project.bronze.product
GROUP BY product_key
HAVING cnt > 1;

In [0]:
%sql
-- Identify inconsistent naming conventions, casing, or missing values in the color column
SELECT DISTINCT color FROM project.bronze.product;

In [0]:
%sql
-- Inspect first and last characters of standard_cost to identify currency symbols and formatting patterns
SELECT DISTINCT 
    LEFT(standard_cost, 1) as start_char,
    RIGHT(standard_cost, 1) as end_char 
FROM project.bronze.product;

In [0]:
%sql
-- Check for NULL product keys 
-- (Ensures data integrity and identifies records that would fail joins in the Gold layer)
SELECT 
    COUNT(*) AS null_product_key_cnt
FROM project.bronze.product
WHERE product_key IS NULL;

Silver Layer: Product Dimension Objective: Cleanse, standardize, and deduplicate the Product Dimension table for Gold Layer consumption.

Key Transformations Deduplication: Applied Key-Level deduplication (dropDuplicates(['product_key'])) to resolve duplicate records and prevent Fan-Out traps in Fact tables. Financial Standardization: Extracted currency (USD, EUR, GBP) and cast standard_cost to Decimal(18,2). String Cleansing: Capitalized color values consistently (e.g., 'MULTI' -> 'Multi') and replaced implicit nulls with 'Unknown'.

In [0]:
from pyspark.sql import functions as F

# Load Bronze Table 
df = spark.table("project.bronze.product")

# Universal String Cleaning
# Trim, remove '|', and map pseudo-nulls to actual NULLs
string_cols = [c for c, t in df.dtypes if t == "string"]
for c in string_cols:
    cleaned = F.trim(F.regexp_replace(F.col(c), r"\|", ""))
    df = df.withColumn(c, 
        F.when(F.upper(cleaned).isin("N/A", "NA", "NULL", "NONE", "-", ""), None)
         .otherwise(cleaned)
    )

# Standardize Colors & Impute Nulls 
if "color" in df.columns:
    # Capitalize first letter consistently (e.g., 'Multi', 'Red')
    df = df.withColumn("color", F.initcap(F.lower(F.col("color"))))
    # Replace remaining Nulls with 'Unknown' for clean BI reporting
    df = df.fillna("Unknown", subset=["color"])

# Split Standard Cost & Currency Detection
if "standard_cost" in df.columns:
    # Detect Currency (Using raw string 'r' to prevent SyntaxWarnings)
    df = df.withColumn("currency",
        F.when(F.col("standard_cost").rlike(r"$"), "USD")
         .when(F.col("standard_cost").rlike("€"), "EUR")
         .when(F.col("standard_cost").rlike("£"), "GBP")
         .otherwise("USD") # Defaulting to USD if no symbol is found
    )
    
    # Clean the string and cast directly to Decimal(18,2) for financial accuracy
    df = df.withColumn("standard_cost_value", 
        F.regexp_replace(F.col("standard_cost"), r"[$,€£\s]", "").cast("decimal(18,2)")
    ).drop("standard_cost")

# Optimization & Deduplication (Prevent Fan-Out Trap) 
# Ensure Keys are Integers
df = df.withColumn("product_key", F.col("product_key").cast("int"))

# Remove null keys and duplicate IDs (Crucial for Dimension Tables)
df = df.filter(F.col("product_key").isNotNull())
df = df.dropDuplicates(["product_key"])

# Metadata & Ordering 
df = df.withColumn("silver_processed_at", F.current_timestamp())

# Ensure ingestion_timestamp exists if not passed from Bronze
if "ingestion_timestamp" not in df.columns:
    df = df.withColumn("ingestion_timestamp", F.current_timestamp())

# Reorder logically for the data warehouse
preferred_order = ["product_key", "product", "standard_cost_value", "currency", "color"]
existing_preferred = [c for c in preferred_order if c in df.columns]
rest = [c for c in df.columns if c not in existing_preferred and c not in ["ingestion_timestamp", "silver_processed_at"]]

df = df.select(*existing_preferred, *rest, "ingestion_timestamp", "silver_processed_at")

# Save to Silver
df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("project.silver.product")

print("Pipeline Status: Success. Silver Table 'silver.product' updated.")
display(df.orderBy("product_key").limit(100))

In [0]:
%sql
SELECT * FROM project.bronze.sales_2017 LIMIT 5;

In [0]:
%sql
SELECT * FROM project.bronze.sales_2018 LIMIT 5;

In [0]:
%sql
SELECT * FROM project.bronze.sales_2019 LIMIT 5;

In [0]:
%sql
SELECT * FROM project.bronze.sales_2020 LIMIT 5;

Simulating Data Inconsistencies (Sabotage for Testing) To demonstrate the robustness of our Silver Layer transformation, we are intentionally introducing common "Legacy System" issues into the 2020 Bronze dataset.

Scenario Objectives:

Schema Evolution: Add a new discount column that only exists in 2020. Date Inconsistency: Change the format of specific rows to a standard ISO format (YYYY-MM-DD) to test multi-format parsing. Data Integrity: Remove a product_key to test how the pipeline handles missing foreign keys.

In [0]:
from pyspark.sql import functions as F

# Load the original 2020 Bronze table
df_2020_raw = spark.table("project.bronze.sales_2020")

print("Original Schema Check (2020):")
df_2020_raw.printSchema()

# Introduce Schema Evolution (The 'discount' column) 
# We simulate a business change where discounts started being recorded in 2020.
# We apply a $5.00 discount only for orders with quantity > 5.
df_2020_sabotaged = df_2020_raw.withColumn("discount", 
    F.when(F.col("quantity") > 5, F.lit("$5.00")).otherwise(F.lit(None))
)

# Introduce Date Inconsistency
# We change a specific order's date to '2020-05-15' (ISO format) 
# to break the 'Thursday, February 27, 2020' pattern.
target_order = "SO63283"
df_2020_sabotaged = df_2020_sabotaged.withColumn("order_date", 
    F.when(F.col("sales_order_number") == target_order, F.lit("2020-05-15"))
     .otherwise(F.col("order_date"))
)

# Introduce Missing Keys (Null Product Key)
# We nullify the product_key for the same order to test Referential Integrity handling.
df_2020_sabotaged = df_2020_sabotaged.withColumn("product_key", 
    F.when(F.col("sales_order_number") == target_order, F.lit(None))
     .otherwise(F.col("product_key"))
)

# Overwrite the Bronze Table 
# We overwrite the table so the Silver Layer script "sees" the dirty data.
df_2020_sabotaged.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("project.bronze.sales_2020")

print("-" * 30)
print("SUCCESS: 2020 Bronze data has been sabotaged for testing purposes.")
print("New Column 'discount' added. Inconsistent dates and null keys injected.")
     
     

In [0]:
%sql
--Sabotage check
SELECT sales_order_number, order_date, product_key, discount 
FROM project.bronze.sales_2020 
WHERE sales_order_number = 'SO63283';
     

In [0]:
%sql
-- Data Integrity Scan: Identify records with missing primary/foreign keys in the Bronze layer.
-- This confirms that the sabotaged record (SO63283) is flagged for cleaning in the Silver layer.
SELECT '2020' as source, sales_order_number, product_key, reseller_key, employee_key
FROM project.bronze.sales_2020
WHERE sales_order_number IS NULL 
   OR product_key IS NULL 
   OR reseller_key IS NULL 
   OR employee_key IS NULL;

In [0]:
%sql
-- Granularity Check: Ensure the combination of Sales Order Number and Product Key is unique.
-- If the result is empty, it confirms there are no duplicate line items in the 2020 Bronze data.
SELECT 
    sales_order_number, 
    product_key, 
    COUNT(*) as cnt
FROM project.bronze.sales_2020
GROUP BY sales_order_number, product_key
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Data Profiling: Compare total line items (rows) vs. unique sales orders.
-- This helps identify the average order density (items per order) in the 2020 dataset.
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sales_order_number) AS distinct_orders
FROM project.bronze.sales_2020;

In [0]:
%sql
-- Primary Key Integrity Check: Compare total rows vs. unique line-item combinations.
-- If both counts match, it confirms there are no duplicate records in the 2020 Bronze table.
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT sales_order_number, product_key) AS distinct_pk_cnt
FROM project.bronze.sales_2020;

In [0]:
%sql
-- NULL Value Profiling: Calculate missing entries for all foreign keys in a single scan.
-- This identifies potential join failures (Data Integrity risks) before processing the Silver layer.
SELECT 
    COUNT(*) - COUNT(sales_order_number) AS null_order_num,
    COUNT(*) - COUNT(product_key) AS null_product_key,
    COUNT(*) - COUNT(reseller_key) AS null_reseller_key,
    COUNT(*) - COUNT(employee_key) AS null_employee_key
FROM project.bronze.sales_2020;

This script executes the unified cleansing and transformation logic for the Sales dataset (2017-2020). It has been refactored to ensure high data quality and system robustness.

Key Operations:

Schema Evolution: Merging datasets with dynamic column support (e.g., handling the discount column from 2020). Advanced Date Parsing: Using coalesce to handle inconsistent date formats (Long-form vs ISO). Financial Precision: Implementing Decimal(18,2) types for all monetary values to ensure accounting accuracy. Data Integrity: Imputing missing product_key entries with -1 and filling null discounts with 0.00. Corrected Deduplication: Preservation of all line items within orders by applying global row deduplication.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

#  Data Ingestion & Merging
s17 = spark.table("project.bronze.sales_2017")
s18 = spark.table("project.bronze.sales_2018")
s19 = spark.table("project.bronze.sales_2019")
s20 = spark.table("project.bronze.sales_2020")

df = s17.unionByName(s18, allowMissingColumns=True) \
        .unionByName(s19, allowMissingColumns=True) \
        .unionByName(s20, allowMissingColumns=True)

# Multi-Currency Detection
df = df.withColumn("currency_code", 
    F.when(F.col("sales").rlike(r"$"), "USD")
     .when(F.col("sales").rlike("€"), "EUR")
     .when(F.col("sales").rlike("£"), "GBP")
     .otherwise("USD")
)

# Universal String Cleansing
string_cols = [c for c, t in df.dtypes if t == "string"]
for c in string_cols:
    cleaned = F.trim(F.regexp_replace(F.col(c), r"\|", ""))
    df = df.withColumn(c, 
        F.when(F.lower(cleaned).isin("n/a", "na", "null", "none", "-"), None)
         .otherwise(cleaned)
    )

# Robust Date Parsing (Spark 3.x Fix)
#  Instead of coalesce (which crashes on mismatch in Spark 3), 
# we use 'when' to check the pattern before parsing.
df = df.withColumn("order_date", 
    F.when(F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), 
           F.to_date(F.col("order_date"), "yyyy-MM-dd"))
     .otherwise(
           F.to_date(F.regexp_replace(F.col("order_date"), r"^[A-Za-z]+,\s*", ""), "MMMM d, yyyy")
     )
)

# Financial Precision
money_mapping = {
    "unit_price": "unit_price_amount",
    "sales": "sales_amount",
    "cost": "cost_amount",
    "discount": "discount_amount"
}

for old_name, new_name in money_mapping.items():
    if old_name in df.columns:
        df = df.withColumn(new_name, 
            F.regexp_replace(F.col(old_name), r"[$,€,£]", "").cast("decimal(18,2)")
        )
        if old_name != new_name:
            df = df.drop(old_name)

#  Optimization (Integer Casting) 
int_cols = ["product_key", "reseller_key", "employee_key", "sales_territory_key", "quantity"]
for c in int_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("int"))

#  Data Integrity & Imputation 
if "discount_amount" in df.columns:
    df = df.fillna(0, subset=["discount_amount"])

df = df.fillna(-1, subset=["product_key"])

# Deduplication & Quality Filters 
df = df.dropDuplicates()
df = df.filter(F.col("quantity") > 0)

# Metadata & Save 
df = df.withColumn("silver_processed_at", F.current_timestamp())

df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("project.silver.sales")

print("Pipeline Status: Success. Silver Table 'silver_sales' updated.")
display(df.limit(10))

In [0]:
%sql
-- Data Reconciliation: Compare total row counts between Silver and Bronze layers.
-- The Silver count should match the Bronze total, or be slightly lower if duplicates were removed.
SELECT 
    (SELECT COUNT(*) FROM project.silver.sales) AS silver_cnt,
    (SELECT 
        (SELECT COUNT(*) FROM project.bronze.sales_2017) + 
        (SELECT COUNT(*) FROM project.bronze.sales_2018) + 
        (SELECT COUNT(*) FROM project.bronze.sales_2019) + 
        (SELECT COUNT(*) FROM project.bronze.sales_2020)
    ) AS bronze_total_cnt;

In [0]:
%sql
-- Post-Transformation Validation: Ensure zero NULL values in critical business columns.
-- All counts should be 0, confirming that data imputation (e.g., product_key = -1) 
-- and cleansing logic were successfully applied in the Silver layer.
SELECT 
    COUNT(*) - COUNT(sales_order_number) AS null_order_num,
    COUNT(*) - COUNT(order_date) AS null_date,
    COUNT(*) - COUNT(product_key) AS null_prod_key,
    COUNT(*) - COUNT(sales_amount) AS null_sales
FROM project.silver.sales;

In [0]:
%sql
-- Cleansing Verification: Confirm that sabotaged records with NULL keys 
-- were successfully mapped to the default value (-1) in the Silver layer.
SELECT COUNT(*) AS sabotaged_rows_fixed
FROM project.silver.sales
WHERE product_key = -1;

# Final check

In [0]:
# Show tables ingested in silver
display(spark.sql("SHOW TABLES IN project.silver"))